# 01. Deep-Learning Foundations: From Equations to PyTorch

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner → research-practical  
**Course:** Deep Learning for Optical Imaging

This notebook builds the mathematical and conceptual foundation needed to understand later code rather than treating PyTorch as a collection of commands.

> **How to study this notebook:** read the explanation first, predict what the code should do, run it, change one parameter, and explain why the result changed.


## Learning objectives

- Explain a neuron mathematically
- Understand tensors and NCHW shapes
- Explain loss, gradients, and optimization
- Distinguish train/validation/test roles
- Recognize overfitting and leakage
- Translate an optical-imaging question into an ML specification


## Mind map

```mermaid
mindmap
  root((Deep learning))
    Function
      Input x
      Parameters theta
      Prediction
    Learning
      Loss
      Gradient
      Optimizer
      Learning rate
    Data
      Train
      Validation
      Test
    Generalization
      Underfit
      Overfit
      Leakage
    Optical imaging
      Images
      Masks
      Restoration targets
      Specimen-level split

```


## 1. What deep learning is

A neural network is a parameterized function:

\[
\hat{y}=f(x;\theta)
\]

- \(x\): input, for example an OCT B-scan or fluorescence image
- \(\theta\): learnable parameters (weights/biases)
- \(\hat{y}\): prediction
- \(y\): target or ground truth

Training chooses \(\theta\) so a **loss function** \(L(\hat y, y)\) becomes small on training data while still generalizing to unseen data.

Deep learning is therefore not magic. It is **function approximation + optimization + statistical generalization**.


## 2. A neuron from the original equation to code

A simple neuron computes:

\[
z = \sum_i w_i x_i + b,\qquad a=\phi(z)
\]

where \(\phi\) is an activation function.

### Translate equation → code

1. Identify inputs: `x`
2. Identify parameters: `w`, `b`
3. Compute weighted sum
4. Apply activation
5. Verify output shape


In [ ]:
import torch

x = torch.tensor([0.2, 0.8, -0.4])
w = torch.tensor([1.5, -0.5, 0.25])
b = torch.tensor(0.1)

z = torch.sum(w * x) + b
a = torch.relu(z)

print("z =", z.item())
print("ReLU(z) =", a.item())


## 3. Why nonlinear activations are necessary

If every layer were only a linear transformation, stacking many layers would still collapse to one linear transformation. Nonlinear activations let the network represent curved, piecewise, and hierarchical relationships.

Common starting points:

| Activation | Typical use | Caution |
|---|---|---|
| ReLU | hidden CNN layers | dead activations possible |
| LeakyReLU | hidden layers | adds a small negative slope |
| GELU | transformers / modern models | slightly more compute |
| Sigmoid | binary probability output | not ideal for deep hidden layers |
| Softmax | mutually exclusive multi-class probabilities | apply along class dimension |


## 4. Tensors and image shapes

PyTorch convention for images is usually:

\[
[N,C,H,W]
\]

- `N`: batch size
- `C`: channels
- `H`: height/rows
- `W`: width/columns

For volumetric imaging, a common convention is `[N, C, D, H, W]`.

A major source of bugs is confusing **scientific channels** with batch or depth dimensions.


In [ ]:
import torch

oct_batch = torch.randn(8, 1, 256, 512)     # 8 grayscale B-scans
rgb_batch = torch.randn(8, 3, 256, 256)     # RGB images
volume_batch = torch.randn(2, 1, 64, 256, 256)

print("OCT:", oct_batch.shape)
print("RGB:", rgb_batch.shape)
print("3-D:", volume_batch.shape)


## 5. Loss functions: what the model is actually asked to optimize

A loss is not just a programming detail. It defines what counts as an error.

### Mean absolute error (L1)

\[
L_{L1}=\frac{1}{N}\sum_i |\hat y_i-y_i|
\]

### Mean squared error (L2/MSE)

\[
L_{MSE}=\frac{1}{N}\sum_i(\hat y_i-y_i)^2
\]

MSE penalizes large errors more strongly. L1 is often less dominated by outliers.

For segmentation, BCE, Dice-type losses, or combinations are common. For virtual staining/restoration, pixel losses alone may not capture structure or perceptual quality.


In [ ]:
import torch
import torch.nn.functional as F

target = torch.tensor([0.0, 1.0, 2.0])
pred = torch.tensor([0.1, 1.4, 1.6])

l1 = F.l1_loss(pred, target)
mse = F.mse_loss(pred, target)

print("L1 :", l1.item())
print("MSE:", mse.item())


## 6. Gradient descent conceptually

Training repeats:

1. Forward pass: compute prediction
2. Loss: quantify error
3. Backward pass: compute gradients \(\partial L/\partial\theta\)
4. Optimizer step: change parameters in a direction that reduces loss
5. Repeat

For simple gradient descent:

\[
\theta_{t+1}=\theta_t-\eta\nabla_\theta L
\]

where \(\eta\) is the learning rate.


In [ ]:
import torch

w = torch.tensor(0.0, requires_grad=True)
x = torch.tensor(2.0)
target = torch.tensor(6.0)

# Model: y_hat = w*x
pred = w * x
loss = (pred - target) ** 2
loss.backward()

print("loss:", loss.item())
print("dL/dw:", w.grad.item())

learning_rate = 0.1
with torch.no_grad():
    w -= learning_rate * w.grad

print("updated w:", w.item())


## 7. Training, validation, and test sets

- **Training set:** updates weights.
- **Validation set:** selects hyperparameters/checkpoints.
- **Test set:** estimates final generalization after decisions are frozen.

If you repeatedly look at test results and change the model, the test set has effectively become another validation set.

### Optical-imaging warning
If 100 patches came from the same tissue specimen, those are **not 100 independent biological samples**. Split at the specimen/patient/animal level before patch extraction whenever possible.


## 8. Overfitting and underfitting

**Underfitting:** model cannot fit training data well.  
**Overfitting:** training performance is excellent but validation performance degrades.

Possible causes of overfitting:
- model too flexible for available independent samples;
- too many training epochs;
- strong dataset biases;
- leakage;
- weak regularization;
- train/validation domains differ.

Do not automatically solve overfitting by adding augmentation. First verify the split and labels.


## 9. Tiny-set overfit test — one of the best debugging tools

Before a long training run, try to overfit 4–16 examples. If the model cannot drive training loss very low on a tiny set, investigate:

- wrong labels;
- output/target shape mismatch;
- incorrect activation;
- loss implementation bug;
- normalization problem;
- optimizer not seeing parameters;
- learning rate too small/large.

A model that cannot memorize a tiny dataset is usually not ready for full training.


## Worked optical-imaging example

**Question:** segment blood vessels in fluorescence images.

Translate the research problem:

```text
Input:     1-channel fluorescence image
Target:    binary vessel mask
Output:    one logit per pixel
Model:     U-Net baseline
Loss:      BCEWithLogits + Dice
Split:     animal-level, not patch-level
Metric:    Dice + sensitivity + specificity
QC:        overlay prediction on original image
Risk:      dim vessels may be systematically missed
```

This specification is more important than deciding whether the encoder has 32 or 64 initial filters.


## End-of-notebook checklist

Before moving on, you should be able to explain the main ideas **without looking at the code**. If you cannot explain why a method, loss, split, or metric is appropriate, repeat the relevant section before using it in research.
